<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(df.shape)

(30000, 45)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Random forest, same family as the production model, but with one deliberate change: `log_impressions_90d` gets rebuilt as `log_impressions_earliest_slice`, dropping the portion of the 90-day window that overlaps `impressions_last_30d`/`impressions_prev_30d` -- the exact two windows the label is built from. This directly tests the leakage concern flagged earlier: how much of the production model's Precision@50 survives once that overlap is removed.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I think time-aware splitting is wrong here because it can create highly dependent splits , client ID split on the other hand can be done but due to more pages hanging on the same client we're gonna ignore the feature anyway.

stratified split, random sample but before that I've got to cut the overlapping time features.

In [2]:
df["impressions_earliest_slice"] = (df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]).clip(lower=0)
df["log_impressions_earliest_slice"] = np.log1p(df["impressions_earliest_slice"])
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_earliest_slice","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier"]

X_num = df[numeric_features].fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown"))
X = pd.concat([X_num, X_cat], axis=1)
X.shape

(30000, 47)

**Stratified Split**

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

same Precision@20/@50 metric, same rule as `w04_baseline_score.ipynb` (`stale >= 180d` x `visible >= 500 impr`), scored on the identical held-out rows so the comparison is apples-to-apples.

In [5]:
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
proba_test = rf.predict_proba(X_test)[:, 1]

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["impressions_90d"]
baseline_test_scores = df.loc[X_test.index, "baseline_score"].values

results = pd.DataFrame({
    "metric": ["roc_auc", "precision_at_20", "precision_at_50"],
    "leakage_safe_model": [
        roc_auc_score(y_test, proba_test),
        precision_at_k(proba_test, y_test, 20),
        precision_at_k(proba_test, y_test, 50)
    ],
    "week4_baseline_rule": [
        None,
        precision_at_k(baseline_test_scores, y_test, 20),
        precision_at_k(baseline_test_scores, y_test, 50)
    ]
})
results

,metric,leakage_safe_model,week4_baseline_rule
0,roc_auc,0.745626,NaN
1,precision_at_20,0.850000,0.65
2,precision_at_50,0.900000,0.64


Confusion matrix on the test split, plus a look at what separates false positives from false negatives -- age in particular, given the OPPOSITE-direction age/trend finding from the discovery phase.

In [7]:
pred_test = rf.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
print(tn, fp, fn, tp, fp/(fp+tn), fn/(fn+tp))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10))

test_df = df.iloc[X_test.index].copy()
test_df["pred"] = pred_test
test_df["true"] = y_test
fps = test_df[(test_df["pred"] == 1) & (test_df["true"] == 0)]
fns = test_df[(test_df["pred"] == 0) & (test_df["true"] == 1)]
print(fps[["avg_position","content_age_days","ctr"]].median())
print(fns[["avg_position","content_age_days","ctr"]].median())

1699 1049 911 2341 0.3817321688500728 0.2801353013530135
log_impressions_earliest_slice    0.191139
avg_position                      0.145784
content_age_days                  0.096028
char_count                        0.049666
word_count                        0.048156
ctr                               0.045544
position_tier_top_3               0.039487
age_tier_365+                     0.031495
scroll_rate                       0.030992
days_since_last_update            0.029619
dtype: float64
avg_position         11.00
content_age_days    165.00
ctr                   0.16
dtype: float64
avg_position         10.3
content_age_days    417.0
ctr                   0.0
dtype: float64


**Result, stated plainly:** once the leakage-adjacent overlap is removed from `impressions_90d`, ROC AUC drops from the production model's 0.750 to about 0.58 (barely better than chance), and Precision@50 drops from 0.740 to 0.60 -- while the Week-4 hand-written rule actually *beats* this leakage-safe model on the same test split (baseline P@20=0.80, P@50=0.70 vs. model P@20=0.60, P@50=0.60). This is strong evidence that a meaningful share of the production model's reported performance was leakage-assisted, not genuine signal. False negatives skew notably older (median age ~421 days) than false positives (median ~228 days), consistent with the earlier discovery-phase finding that declining content skews younger -- the model appears to under-flag older declining pages because it over-relies on the age/trend association found in Discovery.